# 重点拆解VERL里面的GRPO的数据流

原始prompt -》完整 rollout batch → reward → old/ref log_probs → group advantage → policy loss → update_actor


# RayPPOTrainer - fit函数

## 第一阶段：从原始 prompt 到完整 rollout batch

这一阶段的目标是：从 DataLoader 取出 $B$ 个原始 prompt，让每个 prompt 生成 $G=\text{rollout\_n}$ 条 response，最终得到 $B\times G$ 条可以计算 reward 和 loss 的训练样本。

```text
train_dataloader
取出 batch_dict：B 个原始 prompt
        ↓
DataProto.from_single_dict(batch_dict)
得到 batch，batch size = B
        ↓
为每个原始 prompt 分配 uid
A → uid_A，B → uid_B
        ↓
_get_gen_batch(batch)
提取 rollout 生成所需的 raw_prompt、uid、工具参数等
得到 gen_batch，batch size = B
        ↓
gen_batch.repeat(rollout_n, interleave=True)
A,A,...,B,B,...，batch size：B → B × G
uid 同步复制，同一 prompt 的 G 个请求仍共享同一个 uid
        ↓
combined_gen_batch
普通 PPO/GRPO 中，它就是重复后的生成请求
        ↓
async_rollout_manager.generate_sequences(combined_gen_batch)
rollout/vLLM 真正生成 response
        ↓
combined_gen_output
包含 B × G 条生成结果，例如 responses、input_ids、attention_mask 等
        ↓
slice(0, num_sampled_prompts)
取出正常随机采样的结果，得到真正的 gen_batch_output
        ↓
原始 batch.repeat(rollout_n, interleave=True)
让原始 prompt、uid、reward 信息也从 B 行变成 B × G 行
        ↓
batch.union(gen_batch_output)
把重复后的原始信息与生成结果逐行合并
        ↓
完整训练 batch，batch size = B × G
        ↓
compute_response_mask(batch)
有效 response token = 1，padding token = 0
```

#### 用两个 prompt 举例

当 $B=2$、$G=3$ 时：

```text
生成前：
prompt_A → uid_A
prompt_B → uid_B

repeat 后：
prompt_A → uid_A → response_A1
prompt_A → uid_A → response_A2
prompt_A → uid_A → response_A3
prompt_B → uid_B → response_B1
prompt_B → uid_B → response_B2
prompt_B → uid_B → response_B3
```


### 代码块 1：准备 $B\times G$ 个 rollout 生成请求

下面的代码对应上方数据流中的前五步：

1. `train_dataloader` 取出 $B$ 个原始 prompt；
2. `DataProto.from_single_dict()` 将普通字典转换成 `batch`；
3. 为每个原始 prompt 分配唯一的 `uid`；
4. `_get_gen_batch()` 提取 rollout 生成需要的字段；
5. `gen_batch.repeat(rollout_n)` 将每个 prompt 复制 $G$ 次。

**输入：** `batch_dict`，包含 $B$ 个尚未生成 response 的原始 prompt。

**输出：** `gen_batch_output`，包含 $B\times G$ 个生成请求；同一 prompt 的 $G$ 个请求共享相同的 `uid`。

> 注意：这个代码块结束时还没有真正生成 response。下一代码块将从 `generate_sequences()` 开始。


In [1]:

        # 从恢复训练时的 epoch 开始，循环到配置指定的总 epoch 数。
        for epoch in range(current_epoch, self.config.trainer.total_epochs):
            # DataLoader 每次给出一批原始 prompt；此时还没有生成 response。
            for batch_dict in self.train_dataloader:
                # 某些后端支持异步保存 checkpoint；此方法用于处理已完成保存任务的 finalize 回调。
                if hasattr(self.actor_rollout_wg, "async_calls_finalize_fn_exec"):
                    # blocking=False：只检查并收尾“已经完成”的异步保存，不等待仍在写盘的任务。
                    self.actor_rollout_wg.async_calls_finalize_fn_exec(blocking=False)

                # 保存当前训练 step 产生的指标，例如 reward、loss 和序列长度。
                metrics = {}
                # 保存当前训练 step 各阶段的耗时。
                timing_raw = {}

                # 记录“启动性能分析器”这一阶段的耗时；学习数据流时可以先跳过此块。
                with marked_timer("start_profile", timing_raw):
                    # 根据当前 step 是否需要 profiling，启动或保持性能分析器状态。
                    self._start_profiling(
                        # 连续 profiling 模式下，只在从“未分析”切换到“需要分析”时启动。
                        not prev_step_profile and curr_step_profile
                        if self.config.global_profiler.profile_continuous_steps
                        # 非连续模式下，当前 step 需要分析就直接启动。
                        else curr_step_profile
                    )

                # 把 DataLoader 返回的普通字典转换成 VERL 统一传递数据的 DataProto。
                batch: DataProto = DataProto.from_single_dict(batch_dict)
                # temperature 是整个 batch 共用的生成配置，因此放在 meta_info 中。
                batch.meta_info["temperature"] = self.config.actor_rollout_ref.rollout.temperature

                # 为每个“原始 prompt”生成一个唯一 ID；同一 prompt 的多个 response 会共享该 ID。
                batch.non_tensor_batch["uid"] = np.array(
                    # len(batch.batch) 是原始 prompt 数量，因此这里生成同样数量的 UUID 字符串。
                    [str(uuid.uuid4()) for _ in range(len(batch.batch))],
                    # UID 是字符串而不是 Tensor；DataProto 要求逐样本非 Tensor 字段使用 object 数组。
                    dtype=object,
                )

                # 从完整 batch 中挑出 rollout 生成 response 所需的字段，得到生成专用的 gen_batch。
                gen_batch = self._get_gen_batch(batch)

                # 把当前全局训练步数附加到生成 batch，供 trace/日志定位这次 rollout。
                gen_batch.meta_info["global_steps"] = self.global_steps
                # rollout_n 表示每个 prompt 要采样多少条 response，也就是 GRPO 的组大小 G。
                rollout_n = self.config.actor_rollout_ref.rollout.n
                # 将每个 prompt 连续复制 rollout_n 次：A,A,...,B,B,...，准备进行组采样。
                # repeat() 也会同步复制 non_tensor_batch，因此同组 response 会保留相同的 uid。
                gen_batch_output = gen_batch.repeat(repeat_times=rollout_n, interleave=True)

NameError: name 'current_epoch' is not defined

### 代码块 2：生成 response，并拼成完整训练 batch

下面代码把 `combined_gen_batch` 交给 rollout 服务生成 response。这里的输入和输出都是 `DataProto`，不能用一个统一的张量形状描述。

**输入：** `combined_gen_batch`，batch size 为 $N=B\times G$。

**生成后的主要 Tensor 字段：**

- `prompts`：$[N, P]$
- `responses`：$[N, R]$
- `response_mask`：$[N, R]$
- `input_ids`、`attention_mask`：$[N, P+R]$ gen_batch_output里面就有个response的input ids ，然后batch 里面也有一个 然后union的时候直接合并
- `rollout_log_probs`：可选，$[N, R]$ gen_batch_output 生成的时候貌似会计算 

随后复制的是原始 `batch`，再通过 `batch.union(gen_batch_output)` 按字段横向合并；不是把 `gen_batch` 合并回来。


In [ ]:

# PPO/GRPO 不需要 REMAX 的贪心 baseline，直接使用重复后的生成请求。
combined_gen_batch = gen_batch_output
# 非 REMAX 情况下，整个 combined_gen_batch 都属于正常采样请求。
num_sampled_prompts = len(gen_batch_output)

# 开始生成
combined_gen_output = self.async_rollout_manager.generate_sequences(combined_gen_batch)

# 取生成结果的前 num_sampled_prompts 条，得到正常随机采样产生的 response。
gen_batch_output = combined_gen_output.slice(0, num_sampled_prompts)
# __do_sample__ 只是控制生成方式的临时字段，生成结束后不再需要。
if "__do_sample__" in gen_batch_output.non_tensor_batch:
    # 从正常生成结果中删除临时的 __do_sample__ 字段。
    gen_batch_output.pop(non_tensor_batch_keys=["__do_sample__"])

# REMAX 还要处理拼接在后半段的贪心 baseline；PPO/GRPO 可继续跳过本分支。
if self.config.algorithm.adv_estimator == AdvantageEstimator.REMAX:
    # 从 combined_gen_output 的后半段取出每个原始 prompt 的 baseline 回答。
    gen_baseline_output = combined_gen_output.slice(num_sampled_prompts, None)
    # baseline 生成结束后，同样不再需要 __do_sample__ 临时字段。
    if "__do_sample__" in gen_baseline_output.non_tensor_batch:
        # 删除 baseline 结果中的生成控制标记。
        gen_baseline_output.pop(non_tensor_batch_keys=["__do_sample__"])

    # 如果启用了 Reward Model 且 baseline 结果尚无分数，就在这里计算 baseline reward。
    if self.use_rm and "rm_scores" not in gen_baseline_output.batch.keys():
        # 将 baseline 回答送入共置的 Reward Model，得到逐 token 的奖励分数。
        baseline_reward = self._compute_reward_colocate(gen_baseline_output)
        # 把 Reward Model 返回的 rm_scores 合并进 baseline 生成结果。
        gen_baseline_output = gen_baseline_output.union(baseline_reward)

    # 沿 response token 维度求和，得到每条 baseline 回答的总奖励。
    reward_baseline_tensor = gen_baseline_output.batch["rm_scores"].sum(dim=-1)
    # 将每个原始 prompt 的 baseline 总奖励保存回原始 batch，供 REMAX 计算 advantage。
    batch.batch["reward_baselines"] = reward_baseline_tensor

    # baseline 已经转成 reward_baselines，释放其临时生成结果引用。
    del gen_baseline_output
# combined 对象后续不再使用，及时删除引用以降低大 batch 占用的内存。
del combined_gen_batch, combined_gen_output
# 原始 batch 也按相同顺序重复 rollout_n 次，使其行数与多条 response 对齐。
batch = batch.repeat(
    # 每个原始 prompt 复制的次数必须与生成阶段使用的 rollout.n 相同。
    repeat_times=self.config.actor_rollout_ref.rollout.n,
    # interleave=True 保证顺序为 A,A,...,B,B,...，与生成请求的顺序一致。
    interleave=True,
)
# 把重复后的 prompt/奖励信息与 rollout 返回的 response/token 信息合成完整训练 batch。
batch = batch.union(gen_batch_output)

# 某些 rollout 后端会直接返回 response_mask；没有返回时才在 Trainer 中补算。
if "response_mask" not in batch.batch.keys():
    # 从 responses 和 attention_mask 中截取回答区域，标记有效回答 token 与 padding。
    batch.batch["response_mask"] = compute_response_mask(batch)

至此，VERL 已经完成：$B$ 个 prompt → 每个 prompt 生成 $G$ 条 response → 得到 $B\times G$ 条完整 rollout 样本。每条样本同时保留 prompt、response、uid 和 response mask，接下来可以计算 reward。

## 第二阶段：从完整 rollout batch 到 reward tensor

个人大白话总结 ： 用prompt + response [Batch * Group, response_length] 来生成一个 [Batch*Group]的评分， 然后为了后面方便计算给他打包成[batch*group, response_length]

### 代码块 3：计算、合并并取出 reward

下面代码对应三个动作：

1. `_compute_reward_colocate(batch)`：组织 Reward Worker 为 $N=B\times G$ 条 response 计算奖励；
2. `batch.union(batch_reward)`：把 `rm_scores` 等奖励字段横向合并回完整 batch，行数仍为 $N$；
3. `extract_reward(batch)`：取出 `reward_tensor` 和可选的奖励附加信息。

**主要输出：** `reward_tensor`，通常形状为 $[N,R]$。

这里reward_tensor 形状基本和response保持一致， 并且其奖励会放在最后一个有效token的位置
response 0：[北京, 是, 中国首都, PAD, PAD]
rm_scores ：[  0,  0,     1.2,   0,   0]
                              ↑
                         最后一个有效 token

response 1：[答案, 应该, 是, 42, EOS]
rm_scores ：[  0,    0,  0,  0, -0.4]
                                  ↑
                             最后一个有效 token

response 2：[不知道, EOS, PAD, PAD, PAD]
rm_scores ：[   0,   0.7,  0,   0,   0]
                    ↑
               最后一个有效 token

In [ ]:
# 将包含 prompt、response 和 mask 的完整 batch 交给 RewardLoopManager 计算奖励。
batch_reward = self._compute_reward_colocate(batch)
# 按字段横向合并 Reward Model 的输出；batch size 仍保持为 B × G。
batch = batch.union(batch_reward)

# 从 batch["rm_scores"] 取出训练使用的reward_tensor，并收集可选的奖励附加信息。
reward_tensor, reward_extra_infos_dict = extract_reward(batch)

### 代码块 4：`_compute_reward_colocate()` 只是转发层

这个函数本身不计算 reward。它只检查 `RewardLoopManager` 是否存在，然后把完整 batch 转交给 `compute_rm_score()`，最后返回奖励 `DataProto`。

```text
RayPPOTrainer → RewardLoopManager.compute_rm_score() → batch_reward
```

In [ ]:
# 这是 RayPPOTrainer 内部的薄封装：输入完整 rollout batch，输出奖励 DataProto。
def _compute_reward_colocate(self, batch):
    # 如果 RewardLoopManager 尚未初始化，后续无法分发 reward 计算，立即报错。
    assert self.reward_loop_manager is not None
    # 把完整 batch 转交给 RewardLoopManager；真正的切分、并行打分发生在下一层。
    batch_reward = self.reward_loop_manager.compute_rm_score(batch)
    # 返回只包含 rm_scores 和可选奖励附加字段的 DataProto。
    return batch_reward

### `RewardLoopManager.compute_rm_score()`：组织并行 reward 计算

这个函数是 Reward 阶段的调度中心：它切分 batch、调用多个 Ray Reward Worker、收集回答级分数，再把标量分数装配成逐 token 的 `rm_scores`。

#### 你备注中的四个疑问

**1. `data.chunk(worker数量)` 做什么？**

它沿 batch 第 0 维，把 $N=B\times G$ 条样本切成与 Reward Worker 数量相同的若干 `DataProto`。Tensor 字段和非 Tensor 字段会按相同边界一起切分，`meta_info` 会传给每个 chunk。

例如 $N=6$、2 个 worker：`[0,1,2,3,4,5] → [0,1,2] + [3,4,5]`。

**2. 为什么需要 `ray.get()`？**

`worker.compute_score_batch.remote(chunk)` 只会立即返回一个 `ObjectRef`，它是远程结果的引用，不是实际分数。`ray.get([...])` 会等待这些并行任务完成，并把多个 `ObjectRef` 解析成真正的 Python 结果。

**3. 为什么 `outputs_flat` 不能用 `.values()`？**

`outputs` 的类型是 `list[list[dict]]`，外层和内层都是列表；`.values()` 只属于字典。列表推导式是在保持顺序的同时把两层列表展平成 `list[dict]`。也可以用 `itertools.chain.from_iterable(outputs)`，但不能直接用 `.values()`。

**4. `assemble_rm_scores()` 做什么？**

每条 response 最初只有一个标量 `reward_score`，即 `scores` 的形状可理解为 $[N]$。该函数创建全零张量 `rm_scores：[N,R]`，然后把每条样本的标量分数放到该 response 的最后一个有效 token 位置。这样 reward 就能与后续逐 token 的 advantage、mask 和 loss 对齐。

```text
完整 batch：[N,...]
→ 切成 W 个 chunk
→ W 个 Reward Worker 并行打分
→ outputs：list[list[dict]]
→ outputs_flat：list[dict]，长度 N
→ scores：N 个回答级标量
→ assemble_rm_scores()
→ rm_scores：[N,R]
→ 封装并返回 batch_reward
```

In [ ]:
# 输入 data 是完整 rollout batch，batch size 为 N = B × G。
def compute_rm_score(self, data: DataProto) -> DataProto:
    # reward_model_manager 不为空，说明配置了需要单独占用资源的 Reward Model。
    if self.reward_model_manager is not None:
        # 唤醒 Reward Model，使其权重和运行资源进入可计算状态。
        self.reward_model_manager.wake_up()

    # W = len(self.reward_loop_workers) 是 Reward Worker 数量；它们并行处理不同数据块。
    # 沿第 0 维把 N 条样本切成 W 个 DataProto；每个 chunk 的各字段仍保持行对齐。
    # 未启用 padding 时，DataProto.chunk() 要求 N 能被 W 整除。
    chunks = data.chunk(len(self.reward_loop_workers))

    # 每次 .remote(chunk) 都把一个 chunk 发给对应的远程 Reward Worker。
    # .remote() 立即返回 ObjectRef；
    # ray.get() 等待全部任务结束，并取回真正的 Python 结果。
    # 
    # outputs 里面的内容是每个worker处理的结果，然后每个worker都会分配到一定量的样本，然后每个样本都会有个reward_score
    #     outputs = [
    #     # worker 0
    #     [
    #         {"reward_score": 0.8, "reward_extra_info": {...}},
    #         {"reward_score": 0.2, "reward_extra_info": {...}},
    #     ],

    #     # worker 1
    #     [
    #         {"reward_score": -0.3, "reward_extra_info": {...}},
    #         {"reward_score": 0.9, "reward_extra_info": {...}},
    #     ],
    # ]
    
    outputs = ray.get(
        [
            # 每个 worker 返回 list[dict]：chunk 中每条 response 对应一个结果字典。
            worker.compute_score_batch.remote(chunk)
            # strict=True 要求 worker 数与 chunk 数完全相等，避免静默丢失任务。
            for worker, chunk in zip(self.reward_loop_workers, chunks, strict=True)
        ]
    )
    # outputs 的结构是 list[list[dict]]：外层对应 worker，内层对应该 worker 处理的样本。
    # 将两层列表按原顺序展平成 list[dict]，长度恢复为 N。
    # 不能使用 .values()，因为 outputs/sublist 都是 list，只有 dict 才有 .values()。
    outputs_flat = [item for sublist in outputs for item in sublist]

    # 从每条结果字典中取出回答级标量 reward_score，[N]
    scores = [item["reward_score"] for item in outputs_flat]

    # 把每条 response 的标量 reward 放到其最后一个有效 token 位置。
    # 输出 rm_scores 的形状为 [N, response_length]，其他 token 位置为 0。
    #     response 0：[北京, 是, 中国首都, PAD, PAD]
    # rm_scores ：[  0,  0,     1.2,   0,   0]
    #                               ↑
    #                          最后一个有效 token

    # response 1：[答案, 应该, 是, 42, EOS]
    # rm_scores ：[  0,    0,  0,  0, -0.4]
    #                                   ↑
    #                              最后一个有效 token

    # response 2：[不知道, EOS, PAD, PAD, PAD]
    # rm_scores ：[   0,   0.7,  0,   0,   0]
    #                     ↑
    #                最后一个有效 token
    rm_scores = self.reward_manager_cls.assemble_rm_scores(data, scores)
    # 将 rm_scores 包装成 TensorDict；batch_size=len(data)=N 描述其第 0 维样本数。
    batch = TensorDict({"rm_scores": rm_scores}, batch_size=len(data))

    # 某些 reward 函数还会返回判题原因、分项得分等附加信息；没有则使用空字典。
    reward_extra_infos = [output.get("reward_extra_info", {}) for output in outputs_flat]
    # 约定每条样本的附加信息字段一致，因此以第一条结果的 key 作为字段集合。
    reward_extra_keys = list(reward_extra_infos[0].keys())
    # 非 Tensor 附加信息将按字段名保存到这个字典中。
    non_tensor_batch = {}
    # 逐个附加字段收集 N 条样本的值。
    for key in reward_extra_keys:
        # 每个字段转换成长度为 N 的 NumPy 数组，以满足 DataProto 的格式要求。
        non_tensor_batch[key] = np.array([info[key] for info in reward_extra_infos])

    # 独立 Reward Model 完成本批打分后不再立即使用，可以重新休眠以释放资源。
    if self.reward_model_manager is not None:
        # 休眠的是 Reward Model，不会删除刚刚得到的 rm_scores。
        self.reward_model_manager.sleep()

    # 返回一个只携带奖励字段的新 DataProto，后续通过 batch.union(batch_reward) 横向合并。
    return DataProto(
        # Tensor 部分：rm_scores，形状 [N, response_length]。
        batch=batch,
        # 非 Tensor 部分：可选的奖励附加信息，每个字段长度为 N。
        non_tensor_batch=non_tensor_batch,
        # 记录哪些附加字段属于 reward，便于 extract_reward() 按名称取出。
        meta_info={"reward_extra_keys": reward_extra_keys},
    )


核心困惑是：`scores` 原本是“一条 response 一个分数”，为什么又要把它变成 `[N, response_length]`？

假设：

- 一共有 `N=2` 条 response
- 每条最多有 `response_length=5` 个 token
- 两条 response 的最终奖励是：

```python
scores = [0.8, -0.3]
```

但它们的实际长度不同：

```text
response 0：token1 token2 token3 PAD    PAD
response 1：token1 token2 token3 token4 token5
```

对应的有效位置：

```python
response_mask = [
    [1, 1, 1, 0, 0],
    [1, 1, 1, 1, 1],
]
```

### 第一行：把标量奖励放到最后一个有效 token

```python
rm_scores = self.reward_manager_cls.assemble_rm_scores(data, scores)
```

执行后得到：

```python
rm_scores = [
    [0.0, 0.0,  0.8, 0.0,  0.0],
    [0.0, 0.0,  0.0, 0.0, -0.3],
]
```

也就是：

```text
response 0：  0    0    0.8   0    0
                         ↑
                  最后一个有效 token

response 1：  0    0     0    0   -0.3
                                        ↑
                                 最后一个有效 token
```

这里的 `0.8` 和 `-0.3` 仍然是整条 response 的奖励，并不是第三个或第五个 token 单独获得的质量分。

之所以放在最后一个有效 token，是因为这个 reward 表示：

> 整条 response 生成结束后，才得到的最终结果奖励。

这和强化学习里“到达终止状态才获得奖励”很类似。

后续计算 advantage/return 时，最后位置的奖励能够向前传播，影响前面所有生成 token。与此同时，VERL 后面的 PPO/GRPO 计算通常是按照 token 维度进行的，所以要把序列级标量 reward 转换成 token 级形状 `[N, response_length]`。

不能简单放到最后一列，因为 response 可能有 padding。例如第一条 response 的最后一列是 PAD，真正的结束位置是下标 `2`。

---

### 第二行：包装成 `TensorDict`

```python
batch = TensorDict(
    {"rm_scores": rm_scores},
    batch_size=len(data),
)
```

可以把 `TensorDict` 简单理解成：

> 一个专门存放多个 Tensor 字段的字典，并且这些 Tensor 共享相同的 batch 维度。

普通字典类似：

```python
{
    "rm_scores": rm_scores
}
```

包装成 `TensorDict` 后，仍然可以这样取：

```python
batch["rm_scores"]
```

得到：

```python
Tensor([
    [0.0, 0.0, 0.8, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, -0.3],
])
```

这里：

```python
batch_size=len(data)
```

假设 `len(data) == 2`，它表达的是：

```text
这个 TensorDict 里有 2 条 sample
```

不是把数据切成两份，也不是 response 长度为 2。

各个维度分别表示：

```text
rm_scores.shape = [2, 5]
                   ↑  ↑
                   N  response_length
                   │
                   2 条 sample
```

所以这两行代码的完整含义可以概括为：

```text
scores
每条 response 一个标量
形状：[N]
        │
        ▼ assemble_rm_scores
rm_scores
把标量放到该 response 最后一个有效 token
形状：[N, response_length]
        │
        ▼ TensorDict
包装成 VERL/DataProto 能够继续合并和传递的 batch 数据
```

因此，`TensorDict` 这一行只是包装数据，没有再次计算 reward，也没有改变 `rm_scores` 的内容。

## 第三阶段：准备策略概率基准 `old_log_probs` 与 `ref_log_prob`

Reward 阶段结束后，先不要直接跳到 advantage。下一步应当阅读 `ray_trainer.py` 中紧接在 `extract_reward(batch)` 后面的概率计算代码。

这一阶段并不生成新 response，而是让不同策略对**已经生成好的同一批 response**重新打分：

```text
完整 batch：[N=B×G, ...]
        ↓
π_old 对每个 response token 计算 old_log_probs：[N,R]
        ↓
可选：π_ref 对每个 response token 计算 ref_log_prob：[N,R]
        ↓
把概率字段横向 union 回 batch，样本数和顺序不变
        ↓
下一阶段才使用 reward、old/ref 概率计算 KL 和 advantage
```

三个容易混淆的策略：

| 名称 | 表示什么 | 主要用途 |
|---|---|---|
| `π_rollout` | rollout 后端生成 response 时使用的策略 | 产生 response；可选保存 `rollout_log_probs` |
| `π_old` | 本轮 actor 更新前冻结的策略快照 | 生成 PPO ratio 的分母 `old_log_probs` |
| `π_ref` | 长期冻结的参考模型 | 约束策略不要偏离原模型，计算 KL |

> 当前主线先抓住两个输出字段：`old_log_probs:[N,R]` 是 PPO 必需字段；`ref_log_prob:[N,R]` 只在启用 reference policy 时存在。

### 代码块 5：计算并合并 old/reference log probability

下面代码完成第三阶段的四个动作：

1. 判断是否使用 rollout correction 的 bypass 模式；
2. 默认情况下，用更新前 actor 重新计算 `old_log_probs`；
3. 只在有效 response token 上聚合 entropy 日志；
4. 如果启用了 reference policy，再计算并合并 `ref_log_prob`。

**输入：** 已包含 response、`response_mask` 和 reward 的完整 `batch`，batch size 为 $N=B\times G$。

**输出：** `batch` 新增 `old_log_probs:[N,R]`，并可能新增 `ref_log_prob:[N,R]`；没有增加或删除 sample。

In [ ]:
# 读取可选的 rollout correction 配置；普通训练未配置时得到 None。
rollout_corr_config = self.config.algorithm.get("rollout_correction", None)
# 只有配置存在且 bypass_mode=True 时，才直接复用生成阶段的 rollout_log_probs。
bypass_recomputing_logprobs = rollout_corr_config and rollout_corr_config.get("bypass_mode", False)

# 可选分支：不重新运行 actor，而是把 π_rollout 的概率作为 π_old 的概率基准。
if bypass_recomputing_logprobs:
    # 仅在使用该模式时才导入对应处理函数。
    from verl.trainer.ppo.rollout_corr_helper import apply_bypass_mode

    # 在当前 batch 中准备 bypass 模式所需的 old_log_probs 等字段。
    apply_bypass_mode(
        # 已经包含 prompt、response、mask 和 reward 的完整 batch。
        batch=batch,
        # rollout correction 的配置。
        rollout_corr_config=rollout_corr_config,
        # policy loss 配置，用于检查损失形式是否兼容。
        policy_loss_config=self.config.actor_rollout_ref.actor.policy_loss,
    )
# 默认主线：用本轮更新前的 actor 重新计算 π_old。
else:
    # 记录 old_log_probs 计算阶段的耗时。
    with marked_timer("old_log_prob", timing_raw, color="blue"):
        # 对 batch 中已经生成的 response 重新做 actor 前向计算。
        # old_log_prob 主要包含 old_log_probs/entropys:[N,R]；MFU 只用于性能日志。
        old_log_prob, old_log_prob_mfu = self._compute_old_log_prob(batch)
        # 取出每个 response token 的熵，形状通常为 [N,R]。
        entropys = old_log_prob.batch["entropys"]
        # 取出 response 有效 token 掩码，padding 位置为 0，形状为 [N,R]。
        response_masks = batch.batch["response_mask"]
        # 读取熵/loss 的聚合方式和缩放配置。
        actor_config = self.config.actor_rollout_ref.actor
        # 只在有效 response token 上聚合熵，得到一个 batch 级标量。
        entropy_agg = agg_loss(
            # 每个 token 的熵。
            loss_mat=entropys,
            # 排除 response 中的 padding token。
            loss_mask=response_masks,
            # 决定按 token 还是按 sample 等方式聚合。
            loss_agg_mode=actor_config.loss_agg_mode,
            # 应用配置的缩放系数。
            loss_scale_factor=actor_config.loss_scale_factor,
        )
        # 整理 actor 随机程度和本次前向性能指标。
        old_log_prob_metrics = {
            # detach 后转为 Python 标量，只用于日志。
            "actor/entropy": entropy_agg.detach().item(),
            # actor 前向推理的模型浮点利用率。
            "perf/mfu/actor_infer": old_log_prob_mfu,
        }
        # 将这两个指标加入当前 step 的总指标字典。
        metrics.update(old_log_prob_metrics)
        # entropy 已完成日志统计，后续 batch 不需要保存它。
        old_log_prob.batch.pop("entropys")
        # 如果两种 MoE 路由回放模式同时写 routed_experts，就主动报错。
        if "routed_experts" in batch.batch and "routed_experts" in old_log_prob.batch:
            raise ValueError(
                "Detected conflicting router replay configuration: "
                "router_replay.mode='R2' and enable_rollout_routing_replay=True "
                "cannot be enabled simultaneously. "
                "The enable_rollout_routing_replay option is only used in R3 mode; "
                "it should not be set when using R2 mode."
            )
        # 横向合并 old_log_probs；batch size 仍为 N，样本顺序不变。
        batch = batch.union(old_log_prob)
        # rollout 后端也保存了生成时概率时，比较 π_rollout 与 π_old 以做诊断。
        if "rollout_log_probs" in batch.batch.keys():
            # 仅在需要比较时导入调试指标函数。
            from verl.utils.debug.metrics import calculate_debug_metrics

            # 计算两套概率之间的偏差并加入训练日志。
            metrics.update(calculate_debug_metrics(batch))

# 无论走哪个分支，PPO 后续都必须存在 old_log_probs。
assert "old_log_probs" in batch.batch, f'"old_log_prob" not in {batch.batch.keys()=}'

# 只有配置了冻结的参考策略时，才需要计算 π_ref。
if self.use_reference_policy:
    # 记录 reference policy 前向计算的耗时。
    with marked_timer(str(Role.RefPolicy), timing_raw, color="olive"):
        # 让 π_ref 对同一批 response 逐 token 计算 ref_log_prob:[N,R]。
        ref_log_prob = self._compute_ref_log_prob(batch)
        # 横向合并 ref_log_prob；不改变 batch 的样本数和顺序。
        batch = batch.union(ref_log_prob)

## 第四阶段：从 reward 得到 GRPO advantages

这一阶段解决的问题是：Reward Model 只告诉我们“每条 response 得了多少分”，而 actor 更新需要知道“同一个 prompt 的多条 response 中，哪些相对更好、哪些相对更差”。

```text
reward_tensor / rm_scores：[N,R]
        ↓ 保存原始分数
token_level_scores：[N,R]
        ↓ 可选：减去 beta × KL(old, ref)
token_level_rewards：[N,R]
        ↓ 沿 token 维求和
每条 response 的总 reward：scores:[N]
        ↓ 按 uid 恢复 B 个 prompt 组，每组 G 条 response
组内中心化/标准化
        ↓ 把每条 response 的标量 advantage 复制到其全部有效 token
advantages：[N,R]
```

例如同一个 prompt 生成三条 response，reward 为 `[1.0, 0.4, -0.2]`，组均值为 `0.4`。只做中心化时，它们的相对 advantage 为 `[+0.6, 0.0, -0.6]`：第一条会被鼓励，第二条基本不推动参数，第三条会被抑制。

> 第二阶段把标量 reward 放在最后一个有效 token；第四阶段先沿 token 维求和把它取回来，再把计算出的标量 advantage 复制到该 response 的所有有效 token。两次变形的目的不同。

### 代码块 6：准备最终 token reward，并调用 advantage estimator

下面代码对应第四阶段的主调度流程：

1. 可选计算 critic values；纯 GRPO 可跳过；
2. 把 `reward_tensor` 保存为原始 `token_level_scores`；
3. 根据配置决定是否从 reward 中扣除 KL；
4. 可选执行 rollout correction；
5. 调用 `compute_advantage()`，为 batch 增加 `advantages` 和 `returns`。

**输入：** `reward_tensor:[N,R]`，以及 batch 中的 `uid`、`response_mask`、`old_log_probs` 和可选 `ref_log_prob`。

**输出：** `token_level_rewards:[N,R]`、`advantages:[N,R]`、`returns:[N,R]`。

> 普通 GRPO 初读时沿 `use_critic=False → use_kl_in_reward` 的实际配置 → 跳过 rollout correction → `compute_advantage()` 阅读。

KL 放在 reward 中：
reward = score - β × KL(old, ref)
reward/advantage 在本 batch 内固定

KL 放在 actor loss 中：
loss = policy_loss + β × KL(new, ref)
每次 actor 更新都重新计算 KL

In [ ]:
# PPO/GAE 可选分支：critic 估计 values；纯 GRPO 中 self.use_critic 通常为 False。
if self.use_critic:
    # 记录 critic 前向计算耗时。
    with marked_timer("values", timing_raw, color="cyan"):
        # values 通常为 [N,R]，主要供 GAE 使用。
        values = self._compute_values(batch)
        # 横向合并 values，不改变样本数和顺序。
        batch = batch.union(values)

# 记录 reward 整理及 advantage 计算阶段耗时。
with marked_timer("adv", timing_raw, color="brown"):
    # reward 附加信息字典的类型提示。
    reward_extra_infos_dict: dict[str, list]
    # 保存尚未扣除 KL 的原始逐 token 分数，通常就是第二阶段得到的 rm_scores:[N,R]。
    batch.batch["token_level_scores"] = reward_tensor

    # 判题原因、分项得分等附加信息存在时，同步写入 non_tensor_batch。
    if reward_extra_infos_dict:
        # 每个字段转换为长度 N 的 NumPy 数组，保证逐 sample 对齐。
        batch.non_tensor_batch.update({k: np.array(v) for k, v in reward_extra_infos_dict.items()})

    # 路线 A：把 KL 惩罚直接扣在 reward 中。
    if self.config.algorithm.use_kl_in_reward:
        # 内部核心公式：token_level_rewards = token_level_scores - beta * KL(old, ref)。
        batch, kl_metrics = apply_kl_penalty(
            # kl_ctrl 提供 beta；kl_penalty 指定 KL 估计形式。
            batch, kl_ctrl=self.kl_ctrl_in_reward, kl_penalty=self.config.algorithm.kl_penalty
        )
        # 记录平均 KL 和 beta 等指标。
        metrics.update(kl_metrics)
    # 路线 B：不在 reward 中加入 KL，训练 reward 就等于原始 score。
    else:
        # 字段名称改变，但数值不变。
        batch.batch["token_level_rewards"] = batch.batch["token_level_scores"]

    # 可选 rollout correction 分支；普通 GRPO 未配置时整体跳过。
    if (
        # 必须配置 rollout correction。
        rollout_corr_config is not None
        # 必须存在 rollout 生成时记录的概率。
        and "rollout_log_probs" in batch.batch
        # 这里只处理重新计算了 π_old 的 decoupled 模式。
        and not bypass_recomputing_logprobs
    ):
        # 仅启用时导入实现。
        from verl.trainer.ppo.rollout_corr_helper import compute_rollout_correction_and_add_to_batch

        # 计算重要性采样权重、拒绝采样字段及诊断指标。
        batch, is_metrics = compute_rollout_correction_and_add_to_batch(batch, rollout_corr_config)
        # 合并相关日志指标。
        metrics.update(is_metrics)

    # True：原始 GRPO，组内中心化后还除以标准差；False：只减组均值。
    norm_adv_by_std_in_grpo = self.config.algorithm.get(
        # 未配置时默认 True。
        "norm_adv_by_std_in_grpo", True
    )

    # 根据 adv_estimator 选择 GRPO、GAE 等算法，并把 advantages/returns 写回 batch。
    batch = compute_advantage(
        # batch 已包含 token_level_rewards、response_mask 和 uid。
        batch,
        # 当前配置为 grpo 时会进入 GRPO 分支。
        adv_estimator=self.config.algorithm.adv_estimator,
        # gamma 和 lam 主要供 GAE 使用，GRPO 分支不使用。
        gamma=self.config.algorithm.gamma,
        lam=self.config.algorithm.lam,
        # 每个 prompt 生成的 response 数 G。
        num_repeat=self.config.actor_rollout_ref.rollout.n,
        # 控制是否除以组内标准差。
        norm_adv_by_std_in_grpo=norm_adv_by_std_in_grpo,
        # 完整算法配置。
        config=self.config.algorithm,
    )

### 代码块 7：`compute_advantage()` 如何进入真正的 GRPO 公式

`compute_advantage()` 是 estimator 分发层。配置为 GRPO 时，它把四样东西交给 `compute_grpo_outcome_advantage()`：

- `token_level_rewards:[N,R]`：每条 response 的最终训练奖励；
- `response_mask:[N,R]`：标记有效 token；
- `uid:[N]`：恢复“哪些 response 属于同一个 prompt”的分组关系；
- `norm_adv_by_std_in_grpo`：决定组内 reward 是否除以标准差。

真正的 GRPO 核心就是：

$$A_i=\frac{r_i-\operatorname{mean}(r_{group(i)})}{\operatorname{std}(r_{group(i)})+\epsilon}$$

如果关闭标准差归一化，则只有 $A_i=r_i-\operatorname{mean}(r_{group(i)})$。

In [ ]:
# compute_advantage() 中的 GRPO 分支。
elif adv_estimator == AdvantageEstimator.GRPO:
    # 有效 response token 为 1，padding 为 0。
    grpo_calculation_mask = data.batch["response_mask"]

    # 进入 GRPO outcome advantage 的核心实现。
    advantages, returns = core_algos.compute_grpo_outcome_advantage(
        # 最终训练奖励 [N,R]。
        token_level_rewards=data.batch["token_level_rewards"],
        # 有效 token 掩码 [N,R]。
        response_mask=grpo_calculation_mask,
        # uid:[N]；相同 uid 的 G 条 response 构成一个比较组。
        index=data.non_tensor_batch["uid"],
        # 是否除以组内 reward 标准差。
        norm_adv_by_std_in_grpo=norm_adv_by_std_in_grpo,
    )
    # 保存 actor loss 使用的 advantages:[N,R]。
    data.batch["advantages"] = advantages
    # outcome-only GRPO 中 returns 与 advantages 相同。
    data.batch["returns"] = returns

### 代码块 8：`compute_grpo_outcome_advantage()` 的组内相对奖励

这一层才是真正的 GRPO advantage 公式。阅读时抓住三次遍历：

1. 按 `uid` 收集同一 prompt 的 $G$ 个 response reward；
2. 为每组计算 mean/std；
3. 把每条 response 的 reward 改写为组内相对 advantage。

最后的 `unsqueeze(-1) * response_mask` 会把 `[N]` 的 response 级 advantage 扩展为 `[N,R]`：每个有效 token 得到相同 advantage，padding 位置仍为 0。

In [ ]:
# 输入 token_level_rewards:[N,R]，沿 token 维求和得到每条 response 的总 reward:[N]。
scores = token_level_rewards.sum(dim=-1)

# 保存 uid 到同组 reward 列表的映射。
id2score = defaultdict(list)
# 保存每个 uid 组的 reward 均值。
id2mean = {}
# 保存每个 uid 组的 reward 标准差。
id2std = {}

# advantage 是训练目标，不需要为这段统计过程记录梯度。
with torch.no_grad():
    # N=B×G，即当前 response 总数。
    bsz = scores.shape[0]
    # 第一遍：根据 uid 将 N 条 response 放回 B 个 prompt 组。
    for i in range(bsz):
        # 同一 prompt 的 G 个 score 会进入同一个列表。
        id2score[index[i]].append(scores[i])
    # 第二遍：计算每个 prompt 组内的均值和标准差。
    for idx in id2score:
        # 组内只有一条 response 时，使用安全默认值。
        if len(id2score[idx]) == 1:
            id2mean[idx] = torch.tensor(0.0)
            id2std[idx] = torch.tensor(1.0)
        # 正常 GRPO 的 G>1 分支。
        elif len(id2score[idx]) > 1:
            # 将 G 个标量 reward 堆叠为 [G]。
            scores_tensor = torch.stack(id2score[idx])
            # 组内平均 reward。
            id2mean[idx] = torch.mean(scores_tensor)
            # 组内 reward 标准差。
            id2std[idx] = torch.std(scores_tensor)
        # 空组理论上不可能出现。
        else:
            raise ValueError(f"no score in prompt index: {idx}")
    # 第三遍：将绝对 reward 改写为组内相对 advantage。
    for i in range(bsz):
        # 原始 GRPO：减组均值后再除以组标准差。
        if norm_adv_by_std_in_grpo:
            scores[i] = (scores[i] - id2mean[index[i]]) / (id2std[index[i]] + epsilon)
        # Dr.GRPO 风格：只减组均值。
        else:
            scores[i] = scores[i] - id2mean[index[i]]
    # [N] → [N,1] → 乘 [N,R] mask，得到逐有效 token advantage:[N,R]。
    scores = scores.unsqueeze(-1) * response_mask

# outcome-only GRPO 中 advantages 和 returns 相同。
return scores, scores

## 第五阶段：用 GRPO advantage 更新 actor，并同步给 rollout

这是完整数据流的最后一个训练阶段。前四阶段只是在准备训练目标；这一阶段才真正执行 forward、loss、backward 和 `optimizer.step()`，使模型参数发生变化。

```text
batch 中已有：
responses / response_mask / old_log_probs / advantages
        ↓ actor 用当前参数再次前向
new_log_probs = log π_new(response token | context)
        ↓
ratio = exp(new_log_probs - old_log_probs)
        ↓
PPO clipped policy loss × GRPO advantages
        ↓ 可选
entropy bonus + KL(new, ref) loss
        ↓
backward() + optimizer.step()
        ↓
actor 参数从 π_old 更新为 π_new
        ↓
checkpoint_manager.update_weights()
        ↓
把 π_new 同步给 rollout/vLLM，下一轮生成使用新策略
```

这一阶段最关键的四个张量都具有 $[N,R]$ 形状：

| 字段 | 含义 | 是否带梯度 |
|---|---|---|
| `old_log_probs` | 更新前冻结的 $\log\pi_{old}$ | 否 |
| `log_prob` | actor 当前前向得到的 $\log\pi_{new}$ | 是 |
| `advantages` | 第四阶段得到的组内相对优势 | 否 |
| `response_mask` | 有效 response token 标记 | 否 |

> GRPO 改变的是 advantage 的计算方式；actor 更新仍然可以使用 PPO 的 ratio 与 clipping。

### 代码块 9：Trainer 发起 actor 更新，并同步新权重

主循环在 `compute_advantage()` 之后调用 `_update_actor(batch)`。真正的参数更新发生在远程 actor workers；返回的 `actor_output` 只是 loss、clipfrac、KL、grad norm、MFU 等日志指标。

下面省略了可选 checkpoint 保存代码，但该段已在 `ray_trainer.py` 原位置逐行注释。

In [ ]:
# critic warmup 期间只训练 critic，不更新 actor；纯 GRPO 通常不会进入这里。
if self.config.trainer.critic_warmup > self.global_steps:
    # 维持/唤醒 rollout replicas 的权重管理状态。
    self.checkpoint_manager.update_weights(self.global_steps)
# 正常 GRPO 主线：开始更新 actor。
else:
    # 记录 actor forward、loss、backward 和 optimizer.step 的总耗时。
    with marked_timer("update_actor", timing_raw, color="red"):
        # 把完整训练 batch 分发给 actor workers；参数更新在函数内部完成。
        actor_output = self._update_actor(batch)

    # 此处原源码还有可选 checkpoint 保存逻辑。

    # trainer 中的 actor 已经更新，把新权重同步给 rollout/vLLM。
    with marked_timer("update_weights", timing_raw, color="red"):
        # 下一轮 rollout 将使用本轮更新后的 π_new。
        self.checkpoint_manager.update_weights(self.global_steps)

    # 聚合各 actor worker 返回的训练指标。
    actor_output_metrics = reduce_metrics(actor_output.meta_info["metrics"])
    # 加入当前训练 step 的总日志。
    metrics.update(actor_output_metrics)

### 代码块 10：`_update_actor()` 准备 mini-batch 与 PPO epochs

这个函数仍然属于调度层。它把 `DataProto` 转成 worker 使用的 `TensorDict`，配置 mini-batch size、PPO epoch 和 shuffle，然后调用远程 `update_actor()`。

如果配置中的 `ppo_mini_batch_size` 按原始 prompt 数计算，那么实际 response 数要乘组大小 $G=\text{rollout.n}$。例如 8 个 prompt、每个 prompt 4 条 response，对应 32 条训练 sample。

In [ ]:
def _update_actor(self, batch: DataProto) -> DataProto:
    # 读取 rollout 配置。
    rollout_config = self.config.actor_rollout_ref.rollout
    # 记录是否为 multi-turn，供 worker 正确处理 loss mask。
    batch.meta_info["multi_turn"] = rollout_config.multi_turn.enable
    # actor 重算概率时要复用生成温度。
    batch.meta_info["temperature"] = rollout_config.temperature
    # DataProto 转为 worker/engine 使用的 TensorDict。
    batch_td = batch.to_tensordict()
    # 去掉左右 padding 的无效计算，转成 no-padding 表示。
    batch_td = left_right_2_no_padding(batch_td)
    # 配置要求或 entropy 系数非 0 时，actor 前向需要额外计算 entropy。
    calculate_entropy = self.config.actor_rollout_ref.actor.calculate_entropy or (
        self.config.actor_rollout_ref.actor.entropy_coeff != 0.0
    )
    # 蒸馏训练才使用 top-k；普通 GRPO 为 False。
    distillation_use_topk = (
        self.distillation_config.distillation_loss.loss_settings.use_topk
        if is_distillation_enabled(self.config.get("distillation"))
        else False
    )
    # 配置中的 mini-batch size 通常按 prompt 数定义。
    ppo_mini_batch_size = self.config.actor_rollout_ref.actor.ppo_mini_batch_size
    # 一个 prompt 有 G 条 response，所以换算为实际 sample 数时乘 rollout.n。
    ppo_mini_batch_size = ppo_mini_batch_size * self.config.actor_rollout_ref.rollout.n
    # 同一批 rollout 数据被重复训练的 PPO epoch 数。
    ppo_epochs = self.config.actor_rollout_ref.actor.ppo_epochs
    # 内部 DataLoader 随机种子。
    seed = self.config.actor_rollout_ref.actor.data_loader_seed
    # 是否在各 epoch 打乱样本。
    shuffle = self.config.actor_rollout_ref.actor.shuffle
    # 把训练控制参数作为非 Tensor 元信息附加到 batch_td。
    tu.assign_non_tensor(
        batch_td,
        calculate_entropy=calculate_entropy,
        distillation_use_topk=distillation_use_topk,
        global_batch_size=ppo_mini_batch_size,
        mini_batch_size=ppo_mini_batch_size,
        epochs=ppo_epochs,
        seed=seed,
        dataloader_kwargs={"shuffle": shuffle},
        # True 表示执行 loss、backward 和参数更新，而不只是推理。
        compute_loss=True,
    )
    # 分发给 actor workers；内部计算 π_new 并更新参数。
    actor_output = self.actor_rollout_wg.update_actor(batch_td)
    # 只取 worker 返回的训练指标。
    actor_output = tu.get(actor_output, "metrics")
    # 指标增加 actor/ 前缀。
    actor_output = rename_dict(actor_output, "actor/")
    # 统一 MFU 指标命名。
    actor_output["perf/mfu/actor"] = actor_output.pop("actor/mfu")
    # 包装为轻量 DataProto；这里只返回指标，不返回模型权重。
    actor_output = DataProto.from_single_dict(data={}, meta_info={"metrics": actor_output})
    # 新参数已经留在远程 actor workers 内。
    return actor_output

### 代码块 11：Worker 到训练后端的边界

`actor_rollout_wg.update_actor()` 最终调用 worker 的 `update_actor()`。这一层非常薄：它把数据转交给当前 actor engine。

FSDP、Megatron、TorchTitan、VeOmni 等后端分别实现自己的 `train_mini_batch()`、梯度同步和 `optimizer.step()`，所以继续深挖时应根据实际配置只选择一个后端，而不是把所有实现一起读。

In [ ]:
def update_actor(self, data: TensorDict) -> TensorDict:
    # 真正的 mini-batch、forward、loss、backward 和 optimizer.step
    # 由当前配置选择的 actor engine 完成。
    output = self.actor.train_mini_batch(data=data)
    # 指标移到 CPU 后再跨 Ray worker 返回。
    return output.cpu() if output is not None else None

### 代码块 12：GRPO 使用的 PPO clipped policy loss

actor engine 前向得到当前策略的 `log_prob=log π_new` 后，会调用 policy loss。默认 `loss_mode=vanilla` 时，核心公式为：

$$r_t(\theta)=\exp\big(\log\pi_{new}(a_t|s_t)-\log\pi_{old}(a_t|s_t)\big)$$

$$L_t=-\min\left(r_tA_t,\operatorname{clip}(r_t,1-\epsilon,1+\epsilon)A_t\right)$$

代码使用“最小化 loss”的写法，所以先加负号，再在两份 loss 中取较大值。`advantage>0` 的 token 会提高对应 token 概率，`advantage<0` 的 token 会降低对应 token 概率，而 clipping 防止一次更新幅度过大。

In [ ]:
# π_new 与 π_old 的逐 token log probability 差值：[N,R]。
negative_approx_kl = log_prob - old_log_prob
# 避免下一步 exp 数值溢出或下溢。
negative_approx_kl = torch.clamp(negative_approx_kl, min=-20.0, max=20.0)
# ratio = π_new / π_old。
ratio = torch.exp(negative_approx_kl)
# 只在有效 response token 上记录近似 PPO KL。
ppo_kl = verl_F.masked_mean(-negative_approx_kl, response_mask)

# 未裁剪 loss：-ratio × advantage。
pg_losses1 = -advantages * ratio
# 裁剪 ratio 后的 loss。
pg_losses2 = -advantages * torch.clamp(
    ratio, 1 - cliprange_low, 1 + cliprange_high
)
# PPO 的最小化写法：逐 token 取两份 loss 中较大的一个。
clip_pg_losses1 = torch.maximum(pg_losses1, pg_losses2)
# 统计标准 clipping 的触发比例。
pg_clipfrac = verl_F.masked_mean(torch.gt(pg_losses2, pg_losses1).float(), response_mask)

# dual-clip 为负 advantage 样本准备额外边界。
pg_losses3 = -advantages * clip_ratio_c
# 计算 dual-clip 候选结果。
clip_pg_losses2 = torch.min(pg_losses3, clip_pg_losses1)
# 负 advantage 使用 dual-clip，非负 advantage 使用标准 PPO clip。
pg_losses = torch.where(advantages < 0, clip_pg_losses2, clip_pg_losses1)

# rollout correction 启用时应用重要性采样权重。
if rollout_is_weights is not None:
    pg_losses = pg_losses * rollout_is_weights

# 只聚合有效 response token，得到用于 backward 的标量 policy gradient loss。
pg_loss = agg_loss(
    loss_mat=pg_losses,
    loss_mask=response_mask,
    loss_agg_mode=loss_agg_mode,
    **config.global_batch_info,
)

## 五个阶段串起来

```text
阶段 1：B 个 prompt → 每个复制 G 次 → rollout → N=B×G 条 response
阶段 2：每条 response 计算标量 reward → rm_scores:[N,R]
阶段 3：同一批 response 计算 old_log_probs/ref_log_prob:[N,R]
阶段 4：按 uid 恢复 prompt 组 → 组内相对 advantages:[N,R]
阶段 5：π_new/π_old ratio × advantages → clipped loss → 更新 actor → 同步 rollout
```

到这里，一个完整的 VERL GRPO training step 数据流就闭环了：本轮 rollout 产生训练数据，本轮训练得到的新 actor 权重，又会成为下一轮 rollout 使用的策略。